# Interactive-Sentiment-Analysis-Platform

This Colab notebook provides a **functional Gradio application** for interactive sentiment analysis.

It is designed around:
- **text preprocessing**
- **sentiment analysis**
- **keyword-based interpretation**
- **context-aware text modeling**
- **batch and stream-style processing**

The notebook offers a practical and reproducible workflow for exploring sentiment analysis in Google Colab.

In [1]:
# IMPORTANT: run this cell first in a fresh Colab runtime
# Fixes the known Gradio <-> huggingface_hub incompatibility on some Colab images.
!pip -q uninstall -y gradio gradio_client huggingface_hub >/dev/null 2>&1
!pip -q install --no-cache-dir \
    "huggingface_hub==0.25.2" \
    "gradio==4.44.1" \
    "gradio_client==1.3.0" \
    "transformers==4.44.2" \
    "datasets==2.21.0" \
    "accelerate==0.34.2" \
    "scikit-learn>=1.4.2" \
    "pandas>=2.2.2" \
    "numpy>=1.26.4" \
    "matplotlib>=3.8.4" \
    "wordcloud>=1.9.3" \
    "build>=1.2.2"

print("Dependency installation completed.")
print("If Colab reports that a package was already loaded, restart the runtime once and re-run all cells.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 241.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 181.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 248.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 223.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 221.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 259.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 276.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 256.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [2]:
import re
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from transformers import pipeline

warnings.filterwarnings("ignore")

SEED = 42
FAST_MODE = True
random.seed(SEED)
np.random.seed(SEED)

print("Gradio:", gr.__version__)

Gradio: 4.44.1


## Preprocessing

In [3]:
def clean_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^A-Za-z0-9\s'.,!?-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def split_sentences(text: str):
    text = clean_text(text)
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip()]

def simple_keywords(text: str, top_k: int = 10):
    words = [w for w in clean_text(text).split() if len(w) > 2]
    if not words:
        return []
    freq = pd.Series(words).value_counts().head(top_k)
    return [{"token": str(k), "count": int(v)} for k, v in freq.items()]

## Train the transparent TF-IDF branch

In [4]:
ds = load_dataset("glue", "sst2")
train_df = ds["train"].to_pandas()[["sentence", "label"]].rename(columns={"sentence": "text"})
val_df = ds["validation"].to_pandas()[["sentence", "label"]].rename(columns={"sentence": "text"})

if FAST_MODE:
    train_df = train_df.sample(2500, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(500, random_state=SEED).reset_index(drop=True)

train_df["text"] = train_df["text"].map(clean_text)
val_df["text"] = val_df["text"].map(clean_text)

tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("clf", LogisticRegression(max_iter=300, random_state=SEED))
])
tfidf_clf.fit(train_df["text"], train_df["label"])

preds = tfidf_clf.predict(val_df["text"])

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

## DistilBERT contextual branch

In [5]:
hf_sentiment = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    return_all_scores=True,
    truncation=True
)
print("Transformer branch ready.")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Transformer branch ready.


## Hybrid scoring engine

In [6]:
LABEL_MAP = {0: "negative", 1: "positive"}

def _get_tfidf_details(text: str, top_k: int = 10):
    text = clean_text(text)
    vec = tfidf_clf.named_steps["tfidf"]
    clf = tfidf_clf.named_steps["clf"]
    X = vec.transform([text])
    prob = tfidf_clf.predict_proba([text])[0]
    pred = int(np.argmax(prob))

    feature_names = np.array(vec.get_feature_names_out())
    nz = X.nonzero()[1]
    salient = []
    if len(nz) > 0:
        coefs = clf.coef_[0][nz]
        vals = X.data
        contrib = vals * coefs
        order = np.argsort(np.abs(contrib))[::-1]
        for idx in order[:top_k]:
            feat_id = nz[idx]
            salient.append({
                "feature": str(feature_names[feat_id]),
                "weight": float(vals[idx]),
                "contribution": float(contrib[idx])
            })

    return {
        "pred_label": LABEL_MAP[pred],
        "proba_negative": float(prob[0]),
        "proba_positive": float(prob[1]),
        "salient_features": salient
    }

def _get_hf_details(text: str):
    text = clean_text(text)
    scores = hf_sentiment(text[:1500])[0]
    out = {item["label"].lower(): float(item["score"]) for item in scores}
    return {
        "proba_negative": out.get("negative", 0.0),
        "proba_positive": out.get("positive", 0.0)
    }

def hybrid_predict(text: str, alpha: float = 0.65):
    text = clean_text(text)
    if not text:
        return {
            "label": "empty",
            "confidence": 0.0,
            "scores": {"negative": 0.0, "positive": 0.0},
            "keywords": [],
            "salient_features": []
        }

    tfidf_info = _get_tfidf_details(text)
    hf_info = _get_hf_details(text)

    positive = alpha * hf_info["proba_positive"] + (1 - alpha) * tfidf_info["proba_positive"]
    negative = alpha * hf_info["proba_negative"] + (1 - alpha) * tfidf_info["proba_negative"]

    norm = positive + negative + 1e-9
    positive /= norm
    negative /= norm

    label = "positive" if positive >= negative else "negative"
    confidence = max(positive, negative)

    return {
        "label": label,
        "confidence": float(confidence),
        "scores": {"negative": float(negative), "positive": float(positive)},
        "keywords": simple_keywords(text, top_k=10),
        "salient_features": tfidf_info["salient_features"]
    }

hybrid_predict("I absolutely loved this service. It was fast and helpful.")

{'label': 'positive',
 'confidence': 0.8368629072058903,
 'scores': {'negative': 0.16313709179410954, 'positive': 0.8368629072058903},
 'keywords': [{'token': 'absolutely', 'count': 1},
  {'token': 'loved', 'count': 1},
  {'token': 'this', 'count': 1},
  {'token': 'service.', 'count': 1},
  {'token': 'was', 'count': 1},
  {'token': 'fast', 'count': 1},
  {'token': 'and', 'count': 1},
  {'token': 'helpful.', 'count': 1}],
 'salient_features': [{'feature': 'fast',
   'weight': 0.4640386895559004,
   'contribution': 0.220143914837948},
  {'feature': 'absolutely',
   'weight': 0.5051955695164376,
   'contribution': -0.20850356892724192},
  {'feature': 'it',
   'weight': 0.2380330374750571,
   'contribution': -0.11715257806259675},
  {'feature': 'was',
   'weight': 0.38456728327407963,
   'contribution': -0.09681608646990368},
  {'feature': 'this',
   'weight': 0.27760402860706335,
   'contribution': -0.0762134308820226},
  {'feature': 'it was',
   'weight': 0.47228586927152844,
   'contrib

In [7]:
def sentiment_barplot(scores: dict):
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(list(scores.keys()), list(scores.values()))
    ax.set_ylim(0, 1)
    ax.set_title("Hybrid sentiment scores")
    ax.set_ylabel("Probability")
    fig.tight_layout()
    return fig

def feature_contribution_plot(salient_features):
    if not salient_features:
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.text(0.5, 0.5, "No salient TF-IDF features detected", ha="center", va="center")
        ax.axis("off")
        fig.tight_layout()
        return fig

    df = pd.DataFrame(salient_features).head(10)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(df["feature"][::-1], df["contribution"][::-1])
    ax.set_title("Top TF-IDF feature contributions")
    ax.set_xlabel("Contribution to decision")
    fig.tight_layout()
    return fig

def summary_to_markdown(summary: dict) -> str:
    return (
        "### Prediction summary\n"
        f"- **Predicted label:** {summary['predicted_label']}\n"
        f"- **Confidence:** {summary['confidence']:.4f}\n"
        f"- **Negative score:** {summary['negative']:.4f}\n"
        f"- **Positive score:** {summary['positive']:.4f}"
    )

def predict_single(text, alpha):
    result = hybrid_predict(text, alpha=alpha)
    keywords_df = pd.DataFrame(result["keywords"])
    feats_df = pd.DataFrame(result["salient_features"])

    if keywords_df.empty:
        keywords_df = pd.DataFrame(columns=["token", "count"])
    if feats_df.empty:
        feats_df = pd.DataFrame(columns=["feature", "weight", "contribution"])

    summary = {
        "predicted_label": result["label"],
        "confidence": round(result["confidence"], 4),
        "negative": round(result["scores"]["negative"], 4),
        "positive": round(result["scores"]["positive"], 4)
    }
    return summary_to_markdown(summary), sentiment_barplot(result["scores"]), keywords_df, feats_df, feature_contribution_plot(result["salient_features"])

def predict_batch(file_path, text_column, alpha):
    if file_path is None:
        return pd.DataFrame({"error": ["Please upload a CSV file."]}), None

    df = pd.read_csv(file_path)
    if text_column not in df.columns:
        candidates = [c for c in df.columns if df[c].dtype == "object"]
        if not candidates:
            return pd.DataFrame({"error": [f"Column '{text_column}' not found and no text column detected."]}), None
        text_column = candidates[0]

    rows = []
    for text in df[text_column].fillna("").astype(str).tolist():
        res = hybrid_predict(text, alpha=alpha)
        rows.append({
            text_column: text,
            "predicted_label": res["label"],
            "confidence": res["confidence"],
            "negative": res["scores"]["negative"],
            "positive": res["scores"]["positive"]
        })
    out_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(5, 3))
    out_df["predicted_label"].value_counts().plot(kind="bar", ax=ax)
    ax.set_title("Batch prediction distribution")
    ax.set_ylabel("Count")
    fig.tight_layout()
    return out_df, fig

In [8]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

def df_to_markdown(df, max_rows=20):
    if df is None or len(df) == 0:
        return "No data available."
    df2 = df.head(max_rows).copy()
    try:
        return df2.to_markdown(index=False)
    except Exception:
        return df2.to_string(index=False)

def summary_to_markdown(summary: dict) -> str:
    return (
        "### Prediction summary\n"
        f"- **Predicted label:** {summary['predicted_label']}\n"
        f"- **Confidence:** {summary['confidence']:.4f}\n"
        f"- **Negative score:** {summary['negative']:.4f}\n"
        f"- **Positive score:** {summary['positive']:.4f}"
    )

def predict_single(text, alpha):
    if text is None or not str(text).strip():
        empty_fig_1, ax1 = plt.subplots(figsize=(5, 3))
        ax1.text(0.5, 0.5, "No input text provided.", ha="center", va="center")
        ax1.axis("off")
        empty_fig_1.tight_layout()

        empty_fig_2, ax2 = plt.subplots(figsize=(5, 3))
        ax2.text(0.5, 0.5, "No feature contributions available.", ha="center", va="center")
        ax2.axis("off")
        empty_fig_2.tight_layout()

        return (
            "### Prediction summary\n- No text provided.",
            empty_fig_1,
            "No keywords available.",
            "No salient TF-IDF features available.",
            empty_fig_2
        )

    result = hybrid_predict(text, alpha=alpha)

    keywords_df = pd.DataFrame(result.get("keywords", []))
    feats_df = pd.DataFrame(result.get("salient_features", []))

    if keywords_df.empty:
        keywords_df = pd.DataFrame(columns=["token", "count"])
    if feats_df.empty:
        feats_df = pd.DataFrame(columns=["feature", "weight", "contribution"])

    summary = {
        "predicted_label": result["label"],
        "confidence": float(result["confidence"]),
        "negative": float(result["scores"]["negative"]),
        "positive": float(result["scores"]["positive"])
    }

    return (
        summary_to_markdown(summary),
        sentiment_barplot(result["scores"]),
        df_to_markdown(keywords_df),
        df_to_markdown(feats_df),
        feature_contribution_plot(result.get("salient_features", []))
    )

def predict_batch(file_path, text_column, alpha):
    if file_path is None:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, "Please upload a CSV file.", ha="center", va="center")
        ax.axis("off")
        fig.tight_layout()
        return "No CSV uploaded.", fig

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.text(0.5, 0.5, "CSV read error.", ha="center", va="center")
        ax.axis("off")
        fig.tight_layout()
        return f"CSV read error: {e}", fig

    if text_column not in df.columns:
        candidates = [c for c in df.columns if df[c].dtype == "object"]
        if not candidates:
            fig, ax = plt.subplots(figsize=(5, 3))
            ax.text(0.5, 0.5, "No valid text column found.", ha="center", va="center")
            ax.axis("off")
            fig.tight_layout()
            return f"Column '{text_column}' not found and no text column detected.", fig
        text_column = candidates[0]

    rows = []
    for text in df[text_column].fillna("").astype(str).tolist():
        res = hybrid_predict(text, alpha=alpha)
        rows.append({
            text_column: text,
            "predicted_label": res["label"],
            "confidence": round(float(res["confidence"]), 4),
            "negative": round(float(res["scores"]["negative"]), 4),
            "positive": round(float(res["scores"]["positive"]), 4)
        })

    out_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(6, 3))
    out_df["predicted_label"].value_counts().plot(kind="bar", ax=ax)
    ax.set_title("Batch prediction distribution")
    ax.set_ylabel("Count")
    fig.tight_layout()

    return df_to_markdown(out_df, max_rows=30), fig


examples = [
    ["I loved the quality of this product and the support team was amazing."],
    ["This update is terrible, slow, and full of frustrating bugs."],
    ["The service is okay, but the response time could be better."]
]

with gr.Blocks(title="Interactive-Sentiment-Analysis-Platform") as demo:
    gr.Markdown("# Interactive-Sentiment-Analysis-Platform")
    gr.Markdown(
        "Article-inspired hybrid sentiment analysis with preprocessing, TF-IDF explainability, and a DistilBERT contextual branch."
    )

    with gr.Tab("Single text"):
        with gr.Row():
            text_in = gr.Textbox(lines=8, label="Input text")
            alpha_in = gr.Slider(0.0, 1.0, value=0.65, step=0.05, label="Transformer weight (alpha)")

        run_btn = gr.Button("Analyze sentiment")
        summary_out = gr.Markdown()
        with gr.Row():
            score_plot = gr.Plot(label="Sentiment scores")
            feat_plot = gr.Plot(label="Feature contributions")
        with gr.Row():
            keywords_out = gr.Markdown()
            feats_out = gr.Markdown()

        gr.Examples(examples=examples, inputs=[text_in])

        run_btn.click(
            fn=predict_single,
            inputs=[text_in, alpha_in],
            outputs=[summary_out, score_plot, keywords_out, feats_out, feat_plot],
            api_name=False
        )

    with gr.Tab("Batch CSV"):
        file_in = gr.File(label="Upload CSV", file_types=[".csv"], type="filepath")
        text_column_in = gr.Textbox(label="Text column name", value="text")
        alpha_batch = gr.Slider(0.0, 1.0, value=0.65, step=0.05, label="Transformer weight (alpha)")
        batch_btn = gr.Button("Run batch inference")
        batch_table = gr.Markdown()
        batch_plot = gr.Plot(label="Prediction distribution")

        batch_btn.click(
            fn=predict_batch,
            inputs=[file_in, text_column_in, alpha_batch],
            outputs=[batch_table, batch_plot],
            api_name=False
        )

demo.queue()
demo.launch(share=True, debug=False, show_api=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://91fd53d3ef00d30f0d.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
